In [4]:
import os
import re
import pandas as pd

# Paths
PROJECT_ROOT = os.path.abspath("..")  # notebook inside `notebooks/`
RAW_PATH = os.path.join(PROJECT_ROOT, "data", "raw", "sentiment_mental_health.csv")

# Set your text column name here
TEXT_COL = "statement"   # <-- change this if your text column has a different name

# Load raw data
df_raw = pd.read_csv(RAW_PATH)

print("Columns:", df_raw.columns.tolist())
print("Shape:", df_raw.shape)
df_raw.head()


Columns: ['Unnamed: 0', 'statement', 'status']
Shape: (52681, 3)


,Unnamed: 0,statement,status
0,0,oh my gosh,Anxiety
1,1,"trouble sleeping, confused mind, restless hear...",Anxiety
2,2,"All wrong, back off dear, forward doubt. Stay ...",Anxiety
3,3,I've shifted my focus to something else but I'...,Anxiety
4,4,"I'm restless and restless, it's been a month n...",Anxiety


In [5]:
import itertools
from collections import Counter

# Regex pattern to match a wide range of emojis
emoji_pattern = re.compile(
    "["
    "\U0001F600-\U0001F64F"  # emoticons
    "\U0001F300-\U0001F5FF"  # symbols & pictographs
    "\U0001F680-\U0001F6FF"  # transport & map
    "\U0001F1E0-\U0001F1FF"  # flags
    "\U00002702-\U000027B0"
    "\U000024C2-\U0001F251"
    "]",
    flags=re.UNICODE
)

def extract_emojis(text: str):
    """Return a list of emojis found in the given text."""
    if pd.isna(text):
        return []
    text = str(text)
    return emoji_pattern.findall(text)


In [7]:
import unicodedata

def is_visible_emoji(ch: str) -> bool:
    """
    Returns True if `ch` is likely a 'real' emoji and not just a modifier
    like U+FE0F (variation selector) or other non-spacing marks.
    """
    # Ignore the common emoji variation selector explicitly
    if ord(ch) == 0xFE0F:
        return False
    
    cat = unicodedata.category(ch)
    # Many modifiers / invisible codepoints are Mn (Nonspacing_Mark) or Cf (Format)
    if cat in ("Mn", "Cf"):
        return False
    
    return True

def extract_emojis_visible(text: str):
    if pd.isna(text):
        return []
    text = str(text)
    all_found = emoji_pattern.findall(text)
    return [ch for ch in all_found if is_visible_emoji(ch)]


In [8]:
emoji_lists_vis = df_raw[TEXT_COL].dropna().apply(extract_emojis_visible)

from collections import Counter
import itertools

all_emojis_vis = list(itertools.chain.from_iterable(emoji_lists_vis))
occ_counter_vis = Counter(all_emojis_vis)

texts_counter_vis = Counter()
for emojis in emoji_lists_vis:
    for e in set(emojis):
        texts_counter_vis[e] += 1

emoji_stats = pd.DataFrame([
    {
        "emoji": e,
        "total_occurrences": occ_counter_vis[e],
        "num_texts": texts_counter_vis[e],
    }
    for e in occ_counter_vis
]).sort_values("total_occurrences", ascending=False).reset_index(drop=True)

emoji_stats.head(20)


,emoji,total_occurrences,num_texts
0,😭,50,29
1,❤,42,40
2,😔,29,18
3,😂,19,14
4,●,15,4
5,♀,15,11
6,🏴,14,2
7,✨,12,8
8,😅,12,12
9,🏻,11,7


In [9]:
import re
import pandas as pd

# You can add more to this list anytime
EMOTICONS = [
    ":)", "(:", ":-)", ":D", ":-D", ":(", ":-(",
    ";)", ";-)", ":P", ":-P", ":p", ":-p",
    ":O", ":-O", ":o", ":-o",
    ":/", ":-/", ":\\", ":-\\",
    ":'(", ":'-(",
    ":|", ":-|",
    "XD", "xD",
    "-_-", "^_^", "T_T", "T.T", "TT_TT",
    ":')", ":'D"
]

# Build a regex that matches *exactly* these strings (no guessing)
# Sort by length so longer emoticons (e.g. "T_T") match before ":T" parts, etc.
pattern_str = "(" + "|".join(re.escape(e) for e in sorted(EMOTICONS, key=len, reverse=True)) + ")"
emoticon_pattern = re.compile(pattern_str)

def extract_emoticons(text: str):
    """
    Extracts all configured text emoticons from a string.
    Returns a list like [':)', '-_-', ':)'] for that text.
    """
    if pd.isna(text):
        return []
    text = str(text)
    return [m.group(0) for m in emoticon_pattern.finditer(text)]


In [10]:
from collections import Counter
import itertools

# Apply extractor to each text row (use the raw text)
emoticon_lists = df_raw[TEXT_COL].dropna().apply(extract_emoticons)

# Flatten for total occurrences
all_emoticons = list(itertools.chain.from_iterable(emoticon_lists))
emo_occ_counter = Counter(all_emoticons)

# Count in how many different texts each emoticon appears
emo_texts_counter = Counter()
for emos in emoticon_lists:
    for e in set(emos):  # each emoticon counted once per text
        emo_texts_counter[e] += 1

emoticon_stats = pd.DataFrame([
    {
        "emoticon": e,
        "total_occurrences": emo_occ_counter[e],
        "num_texts": emo_texts_counter[e],
    }
    for e in emo_occ_counter
])

emoticon_stats = emoticon_stats.sort_values(
    by="total_occurrences", ascending=False
).reset_index(drop=True)

emoticon_stats.head(20)


,emoticon,total_occurrences,num_texts
0,:/,1201,894
1,:(,486,458
2,:),428,395
3,:D,49,48
4,:'),37,37
5,XD,32,22
6,;),16,16
7,:'(,14,13
8,-_-,12,12
9,:-(,12,11


In [11]:
!pip install -q emoji


In [12]:
import emoji
import pandas as pd

print("emoji version:", emoji.__version__)


emoji version: 2.15.0


In [13]:
def extract_emojis_emoji_lib(text: str):
    """
    Use the `emoji` library to extract all emoji sequences from a string.
    Each emoji (including multi-codepoint ones) is returned once as a string.
    """
    if pd.isna(text):
        return []
    text = str(text)
    
    # emoji.emoji_list returns a list of dicts: {'emoji': '...', 'match_start': i, 'match_end': j}
    try:
        matches = emoji.emoji_list(text)
    except AttributeError:
        # Fallback for older versions: use emoji_list from emoji.core
        matches = emoji.core.emoji_list(text)
    
    return [m["emoji"] for m in matches]


In [15]:
from collections import Counter
import itertools

# Apply extractor to each text row in the RAW dataset
emoji_lists = df_raw[TEXT_COL].dropna().apply(extract_emojis_emoji_lib)

# Flatten for total occurrences
all_emojis = list(itertools.chain.from_iterable(emoji_lists))
emoji_occ_counter = Counter(all_emojis)

# Count in how many different texts each emoji appears
emoji_texts_counter = Counter()
for emojis in emoji_lists:
    for e in set(emojis):  # count once per text
        emoji_texts_counter[e] += 1

emoji_stats = pd.DataFrame([
    {
        "emoji": e,
        "total_occurrences": emoji_occ_counter[e],
        "num_texts": emoji_texts_counter[e],
    }
    for e in emoji_occ_counter
])

emoji_stats = emoji_stats.sort_values(
    by="total_occurrences", ascending=False
).reset_index(drop=True)

emoji_stats


,emoji,total_occurrences,num_texts
0,™,250,196
1,©,97,73
2,😭,50,29
3,😔,29,18
4,❤️,27,25
...,...,...,...
131,💅,1,1
132,🥹,1,1
133,🖕,1,1
134,🤷🏽,1,1


In [16]:
import os

# If these are already defined in your notebook, this will just reuse them.
PROJECT_ROOT = os.path.abspath("..")
DATA_DIR = os.path.join(PROJECT_ROOT, "data")
PROCESSED_DIR = os.path.join(DATA_DIR, "processed")
os.makedirs(PROCESSED_DIR, exist_ok=True)

EMOJI_CSV_PATH = os.path.join(PROCESSED_DIR, "emoji_stats.csv")

emoji_stats.to_csv(EMOJI_CSV_PATH, index=False)
print("Emoji stats saved to:\n", EMOJI_CSV_PATH)
print("Rows:", emoji_stats.shape[0])


Emoji stats saved to:
 /Users/solaris003/Repository/Fall'25/S-NLP/MindScope-AI/data/processed/emoji_stats.csv
Rows: 136


In [17]:
EMOTICON_CSV_PATH = os.path.join(PROCESSED_DIR, "emoticon_stats.csv")
emoticon_stats.to_csv(EMOTICON_CSV_PATH, index=False)
print("Emoticon stats saved to:\n", EMOTICON_CSV_PATH)
print("Rows:", emoticon_stats.shape[0])


Emoticon stats saved to:
 /Users/solaris003/Repository/Fall'25/S-NLP/MindScope-AI/data/processed/emoticon_stats.csv
Rows: 24
